# BirdCLEF+ 2026 — Baseline Submission

Strategy: EfficientNet-B0 (pretrained, frozen) as feature extractor → train linear head on balanced subset  
Metric: Macro ROC-AUC (skips classes with no true positive labels)  
Target runtime: ~15-20 min on Kaggle CPU (limit: 90 min)

In [ ]:
import os, ast, time, warnings, multiprocessing
warnings.filterwarnings('ignore')
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm
from tqdm import tqdm

print(f'PyTorch  : {torch.__version__}')
print(f'CPU cores: {multiprocessing.cpu_count()}')

## Config

In [ ]:
IS_KAGGLE   = Path('/kaggle').exists()
DATA_DIR    = Path('/kaggle/input/birdclef-2026') if IS_KAGGLE else Path('../data/raw')
WORKING_DIR = Path('/kaggle/working')             if IS_KAGGLE else Path('../outputs/submission')
MEL_CACHE   = WORKING_DIR / 'mels'
MODEL_PATH  = WORKING_DIR / 'best_model.pth'
SUB_PATH    = WORKING_DIR / 'submission.csv'
WORKING_DIR.mkdir(parents=True, exist_ok=True)
MEL_CACHE.mkdir(parents=True, exist_ok=True)

SR, DURATION        = 32000, 5.0
N_MELS, N_FFT       = 128, 1024
HOP_LENGTH          = 320
FMIN, FMAX          = 20.0, 16000.0
MAX_PER_CLASS       = 20      # balanced subset
MIN_RATING          = 0.0     # set >0 to filter low-quality recordings
EPOCHS              = 10      # head-only: fast
BATCH_SIZE          = 64
LR                  = 1e-3
NUM_WORKERS         = min(4, multiprocessing.cpu_count())
MODEL_NAME          = 'efficientnet_b0'
SEED                = 42
torch.manual_seed(SEED); np.random.seed(SEED)

print(f'Data dir : {DATA_DIR}  exists={DATA_DIR.exists()}')

## 1. Load Metadata & Balance Dataset

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv')
df['primary_label'] = df['primary_label'].astype(str)
df['secondary_labels'] = df['secondary_labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

species_list = sorted(df['primary_label'].unique().tolist())
NUM_CLASSES  = len(species_list)
s2i          = {s: i for i, s in enumerate(species_list)}
print(f'Total recordings: {len(df):,}  |  Species: {NUM_CLASSES}')

# Balanced subset: up to MAX_PER_CLASS per species, prefer higher ratings
df_sorted = df.sort_values('rating', ascending=False)
df_bal = (df_sorted.groupby('primary_label', group_keys=False)
          .apply(lambda g: g.head(MAX_PER_CLASS)))
df_bal = df_bal.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 80/20 split
n_val  = max(1, int(len(df_bal) * 0.2))
val_df = df_bal.iloc[:n_val]
trn_df = df_bal.iloc[n_val:]
print(f'Balanced: {len(df_bal)} samples  |  Train: {len(trn_df)}  Val: {len(val_df)}')
print(f'Species represented: {df_bal["primary_label"].nunique()} / {NUM_CLASSES}')

## 2. Pre-compute Mels (balanced subset only)

In [ ]:
def load_and_mel(args):
    filename, audio_dir, cache_dir = args
    cache_path = cache_dir / (filename.replace('/', '__').replace('.ogg', '.npy'))
    if cache_path.exists():
        return True
    try:
        wav, native_sr = sf.read(str(audio_dir / filename), dtype='float32')
        if wav.ndim > 1: wav = wav[:, 0]
        if native_sr != SR:
            wav = librosa.resample(wav, orig_sr=native_sr, target_sr=SR)
        target_len = int(SR * DURATION)
        if len(wav) <= target_len:
            wav = np.pad(wav, (0, target_len - len(wav)))
        else:
            start = np.random.randint(0, len(wav) - target_len)
            wav = wav[start: start + target_len]
        mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=N_MELS,
            n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        np.save(cache_path, mel_db.astype(np.float32))
        return True
    except Exception:
        return False

t0 = time.time()
audio_dir = DATA_DIR / 'train_audio'
args_list = [(r['filename'], audio_dir, MEL_CACHE) for _, r in df_bal.iterrows()]
already   = sum(1 for _ in MEL_CACHE.glob('*.npy'))
print(f'Already cached: {already} / {len(df_bal)}')

if already < len(df_bal):
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = {ex.submit(load_and_mel, a) for a in args_list}
        for _ in tqdm(as_completed(futs), total=len(futs), desc='Mel cache'):
            pass

print(f'Done in {time.time()-t0:.0f}s  |  Cached: {sum(1 for _ in MEL_CACHE.glob("*.npy"))}')

## 3. Extract Backbone Features (once, no grad)

In [ ]:
backbone = timm.create_model(MODEL_NAME, pretrained=True, in_chans=1, num_classes=0)
backbone.eval()
FEAT_DIM = backbone.num_features
print(f'Backbone: {MODEL_NAME}  |  Feature dim: {FEAT_DIM}')

def extract_features(df_split):
    feats, labels_list = [], []
    for i in tqdm(range(0, len(df_split), BATCH_SIZE), desc='Extract feats'):
        batch_rows = df_split.iloc[i: i + BATCH_SIZE]
        mels = []
        for _, row in batch_rows.iterrows():
            p = MEL_CACHE / (row['filename'].replace('/', '__').replace('.ogg', '.npy'))
            mels.append(np.load(p))
        mel_t = torch.from_numpy(np.stack(mels)).unsqueeze(1)
        with torch.no_grad():
            feats.append(backbone(mel_t).numpy())
        lbl = np.zeros((len(batch_rows), NUM_CLASSES), dtype=np.float32)
        for j, (_, row) in enumerate(batch_rows.iterrows()):
            if row['primary_label'] in s2i:
                lbl[j, s2i[row['primary_label']]] = 1.0
            for s in row['secondary_labels']:
                if s in s2i: lbl[j, s2i[s]] = 1.0
        labels_list.append(lbl)
    return np.concatenate(feats), np.concatenate(labels_list)

t0 = time.time()
trn_feats, trn_labels = extract_features(trn_df)
val_feats, val_labels = extract_features(val_df)
print(f'Features extracted in {time.time()-t0:.0f}s')
print(f'Train features: {trn_feats.shape}  Val: {val_feats.shape}')

## 4. Train Linear Head

In [ ]:
from torch.utils.data import TensorDataset

def macro_roc_auc(targets, preds):
    scores = [roc_auc_score(targets[:, i], preds[:, i])
              for i in range(targets.shape[1]) if targets[:, i].sum() > 0]
    return float(np.mean(scores)) if scores else 0.0

trn_ds = TensorDataset(torch.from_numpy(trn_feats), torch.from_numpy(trn_labels))
val_ds = TensorDataset(torch.from_numpy(val_feats), torch.from_numpy(val_labels))
trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

head      = nn.Linear(FEAT_DIM, NUM_CLASSES)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(head.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    head.train()
    trn_loss = 0.0
    for feats_b, lbl_b in trn_loader:
        optimizer.zero_grad()
        loss = criterion(head(feats_b), lbl_b)
        loss.backward(); optimizer.step()
        trn_loss += loss.item()
    trn_loss /= len(trn_loader)

    head.eval()
    all_preds, all_tgts = [], []
    with torch.no_grad():
        for feats_b, lbl_b in val_loader:
            all_preds.append(torch.sigmoid(head(feats_b)).numpy())
            all_tgts.append(lbl_b.numpy())
    auc = macro_roc_auc(np.concatenate(all_tgts), np.concatenate(all_preds))
    scheduler.step()

    print(f'Ep {epoch:02d} | train_loss={trn_loss:.4f} | val_auc={auc:.4f}')
    if auc > best_auc:
        best_auc = auc
        torch.save({'backbone': MODEL_NAME, 'head': head.state_dict(),
                    'species_list': species_list}, MODEL_PATH)

print(f'Best val AUC: {best_auc:.4f}')

## 5. Inference on Test Soundscapes

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location='cpu')
head.load_state_dict(ckpt['head'])
backbone.eval(); head.eval()

test_dir   = DATA_DIR / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

def mel_from_wave(wav):
    mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=N_MELS,
        n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db.astype(np.float32)

def predict_soundscape(audio_path):
    wav, native_sr = sf.read(str(audio_path), dtype='float32')
    if wav.ndim > 1: wav = wav[:, 0]
    if native_sr != SR:
        wav = librosa.resample(wav, orig_sr=native_sr, target_sr=SR)

    target_len = int(SR * DURATION)
    windows, end_secs = [], []
    for start in range(0, len(wav) - target_len + 1, target_len):
        windows.append(wav[start: start + target_len])
        end_secs.append(int((start + target_len) / SR))
    if not windows:
        windows.append(np.pad(wav, (0, target_len - len(wav))))
        end_secs.append(5)

    rows = []
    for i in range(0, len(windows), BATCH_SIZE):
        batch_wav = windows[i: i + BATCH_SIZE]
        mels = torch.from_numpy(np.stack([mel_from_wave(w) for w in batch_wav])).unsqueeze(1)
        with torch.no_grad():
            probs = torch.sigmoid(head(backbone(mels))).numpy()
        stem = audio_path.stem
        for probs_row, end_s in zip(probs, end_secs[i: i + BATCH_SIZE]):
            row = {'row_id': f'{stem}_{end_s}'}
            for j, sp in enumerate(species_list):
                row[sp] = float(probs_row[j])
            rows.append(row)
    return rows

all_rows = []
for fp in tqdm(test_files, desc='Inference'):
    all_rows.extend(predict_soundscape(fp))

submission = pd.DataFrame(all_rows)
submission.to_csv(SUB_PATH, index=False)
print(f'Saved {len(submission):,} rows to {SUB_PATH}')
submission.head(3)

## 6. Validate Submission

In [ ]:
sample_sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
missing_cols = set(sample_sub.columns) - set(submission.columns)
print(f'Shape   : {submission.shape}')
print(f'Missing : {missing_cols if missing_cols else "none"}')
print(f'Scores in [0,1]: {((submission[species_list] >= 0) & (submission[species_list] <= 1)).all().all()}')
print('submission.csv ready.')